# Business Case: Revenue + Retention Analysis

This notebook combines:
1. **Short-term revenue impact** (from Notebook 02)
2. **Long-term retention impact** (churn analysis)
3. **Final recommendation** with dollar projections

The key insight: for the within-10% segment, short-term revenue is barely positive,
but the retention benefit (golden handcuff) flips the recommendation.

In [ ]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve().parent / 'src'))

from analysis.churn_analysis import analyze_churn, compute_ltv_impact
from analysis.durability import annualization_scenarios, compute_weekly_effects, estimate_decay_rate

df = pd.read_csv(Path('..') / 'data' / 'processed' / 'experiment_summary.csv')
post_activity = pd.read_csv(Path('..') / 'data' / 'raw' / 'post_period_activity.csv')
assignment = pd.read_csv(Path('..') / 'data' / 'raw' / 'assignment.csv')

## 1. Churn Analysis

### Causal comparison (treatment vs control)
This IS causal because of randomization.

In [ ]:
churn_results = analyze_churn(df)

causal = churn_results['causal']
print('=== CAUSAL: Treatment vs Control ===')
print(f"Treatment churn: {causal['treatment_churn_rate']:.1%}")
print(f"Control churn:   {causal['control_churn_rate']:.1%}")
print(f"Reduction:       {causal['absolute_reduction']:.1%} ({causal['relative_reduction']:.0%} relative)")
print(f"p-value:         {causal['p_value_fisher']:.4f}")
print(f"Churns prevented: ~{causal['n_churns_prevented']:.0f} per 6 months")

In [ ]:
obs = churn_results['observational']
print('=== OBSERVATIONAL: Crossed Tier vs Did Not (within treatment) ===')
print(f"Crossed tier churn:     {obs['crossed_churn_rate']:.1%} (n={obs['n_crossed']})")
print(f"Did not cross churn:    {obs['not_crossed_churn_rate']:.1%} (n={obs['n_not_crossed']})")
print(f"\n⚠ {obs['CAUTION']}")

In [ ]:
if churn_results['iv_estimate']:
    iv = churn_results['iv_estimate']
    print('=== IV ESTIMATE: Causal effect of CROSSING a tier ===')
    print(iv['interpretation'])

## 2. LTV Impact

In [ ]:
ltv = compute_ltv_impact(churn_results)
print(ltv['interpretation'])

## 3. Final Recommendation Table

Combining short-term revenue + long-term retention by segment.

In [ ]:
# TODO: Build the final decision table
# For each distance bucket:
#   - Short-term net revenue impact ($)
#   - Churn reduction (causal)
#   - Retention value ($)
#   - Recommendation (SHOW / SHOW as retention play / OPTIONAL)
#   - Projected annual impact under conservative assumptions

# This is the slide you present to the Head of DS.

## 4. Annualization Scenarios

In [ ]:
weekly = compute_weekly_effects(post_activity, assignment)

# Calculate total incremental revenue from experiment quarter
treatment_df = df[df['group'] == 'treatment']
control_df = df[df['group'] == 'control']
q_incremental = treatment_df['revenue_change'].sum() - control_df['revenue_change'].sum()

scenarios = annualization_scenarios(weekly, q_incremental)
scenarios